# Tổng hợp và kết luận

Mọi bảng số liệu trong notebook này được **sinh ra bằng cách chạy lại toàn bộ 6 bài
× 3 chiều**, không gõ tay. Chạy lại notebook là số liệu tự cập nhật theo code hiện
có trong kho.

Điều kiện tiên quyết: mỗi bài đã qua **kiểm chứng chéo** ở notebook phần tương ứng
— các chiều cùng biến thể phải ra cùng một nghiệm tối ưu. Nhanh hơn mà giải sai bài
thì không nói lên điều gì.

In [1]:
import sys; sys.path.insert(0, "../tools")
from nbutil import *

## 1. Bản đồ nguồn — khoảng trống của từng nền tảng

Bảng này sinh từ các `manifest.json`, nên luôn khớp với code thật trong kho.

In [2]:
show_source_matrix()

| Bài | Tên | OPL | DOcplex.cp | OR-Tools |
|---|---|---|---|---|
| 1.1 | Tô màu đồ thị (Graph Coloring) | ✅ | ✅ | ✍️ |
| 1.2 | N-Queens | ✍️ | ✅ | ✅ |
| 2.1 | Job-shop scheduling (instance ft06) | ✅ | ✅ | ✍️ |
| 2.2 | Xếp ca nhân sự có nguyện vọng (Employee / Shift Scheduling with shift requests) | ✍️ | ✍️ | ✅ |
| 3.1 | Lập lịch thi đấu thể thao (double round-robin) | ✅ | ✅ | ✍️ |
| 3.2 | Xếp thời khoá biểu có ràng buộc khả dụng | ✅+✍️ | ✍️ | ✍️ |

✅ lấy mẫu chính thức · ✍️ viết mới · ✅+✍️ mẫu chính thức có mở rộng · — chưa có

Đọc bảng theo cột chứ không theo hàng:

- **OR-Tools** thiếu mẫu chính thức ở 4/6 bài — nhưng vì kho ví dụ của Google mỏng
  hơn, không phải vì CP-SAT làm không được. Cả 4 bản viết mới đều chạy tốt.
- **Hệ IBM** thiếu đúng một chỗ có ý nghĩa: **bài 2.2 xếp ca nhân sự bằng CP**.
  Cả OPL lẫn `docplex.cp` đều không có. Bản `nurses` của IBM là MILP chạy trên
  engine CPLEX — bằng chứng ở mục (c) của bài 2.2.
- Bài **3.2** không nền tảng nào có sẵn phần ràng buộc khả dụng.

## 2. Trục NGÔN NGỮ — OPL vs DOcplex.cp

Hai chiều **cùng chạy engine CP Optimizer**, nên mọi chênh lệch ở đây là do ngôn
ngữ mô hình hoá, không phải do engine.

In [3]:
lang, eng = axis_tables()
lang

,bài,tên,OPL nhánh,DOcplex.cp nhánh,OPL (s),DOcplex.cp (s),tỉ lệ nhánh
0,1.1,Tô màu đồ thị (Graph Coloring),202,201,0.020,0.023,1.00
1,1.2,N-Queens,440,255,0.023,0.023,0.58
2,2.1,Job-shop scheduling (instance ft06),140843,141455,0.039,0.037,1.00
3,2.2,Xếp ca nhân sự có nguyện vọng (Employee / Shif...,1133,1325,0.015,0.013,1.17
4,3.1 (A),Lập lịch thi đấu thể thao (double round-robin),556103,850352,1.556,2.313,1.53
5,3.2,Xếp thời khoá biểu có ràng buộc khả dụng,125151,154818,7.030,1.319,1.24


### Đọc bảng trên

Cột `tỉ lệ nhánh` là *nhánh DOcplex.cp ÷ nhánh OPL*. Bằng 1.00 nghĩa là hai ngôn
ngữ đưa cho engine đúng cùng một bài toán.

**Kết luận chính của trục này:** ngôn ngữ **chỉ** ảnh hưởng tới hiệu năng khi nó
ép người viết dựng một mô hình toán khác đi.

| Nhóm | Bài | Vì sao |
|---|---|---|
| Tỉ lệ ≈ 1.00 | 1.1, 2.1 | mọi khái niệm ánh xạ một-một giữa hai ngôn ngữ; engine nhận đúng cùng một bài |
| Lệch nhẹ (1.1–1.6×) | 2.2, 3.1 | cùng mô hình toán, khác **thứ tự nạp biến/ràng buộc** xuống engine |
| Lệch lớn (0.58×) | 1.2 | `allDifferent` của OPL không nhận mảng biểu thức ⇒ **ép thêm $2n$ biến phụ** |

Hai bài lệch nhẹ đã được **truy nguyên nhân bằng thí nghiệm đối chứng**, không phải
suy đoán: ở bài 1.1, đảo vế đúng một ràng buộc làm số nhánh nhảy 201 → 202 và ra
đúng nghiệm của chiều OPL; ở bài 2.2, hoán vị thứ tự post ràng buộc làm 1 325 tụt
về 1 118. Cả hai đều không đổi mô hình toán.

> ⚠️ **Hàng 3.2 không phải phép so ngôn ngữ thuần.** Ở bài đó hai chiều cố ý dùng
> hai cách mã hoá khác nhau (OPL đếm có điều kiện, DOcplex.cp biến interval), nên
> chênh lệch trộn cả ngôn ngữ lẫn mã hoá. Đó chính là điều bài 3.2 muốn đo — xem
> mục 5.

Một điểm đáng chú ý về chiều: bài 1.2 OPL **thua**, nhưng bài 3.1 OPL **thắng**
1.5×. OPL không hề "chậm hơn"; nó gọn hơn ở chỗ có `tupleset` với khoá, `inverse`
theo range khai báo, và vướng ở chỗ ràng buộc toàn cục không nhận biểu thức.

## 3. Trục ENGINE — CP Optimizer vs CP-SAT

Hai chiều **cùng viết bằng Python**, nên chênh lệch ở đây là do engine.

In [4]:
eng

,bài,tên,CPO nhánh,CP-SAT nhánh,CPO (s),CP-SAT (s),CP-SAT ít nhánh hơn,engine nhanh hơn
0,1.1,Tô màu đồ thị (Graph Coloring),201,37,0.023,0.0017,5.43,CP-SAT
1,1.2,N-Queens,255,475,0.023,0.0099,0.54,CP-SAT
2,2.1,Job-shop scheduling (instance ft06),141455,219,0.037,0.0110,645.91,CP-SAT
3,2.2,Xếp ca nhân sự có nguyện vọng (Employee / Shif...,1325,254,0.013,0.0051,5.22,CP-SAT
4,3.1 (A),Lập lịch thi đấu thể thao (double round-robin),850352,1602,2.313,0.6962,530.81,CP-SAT
5,3.2,Xếp thời khoá biểu có ràng buộc khả dụng,154818,5748,1.319,2.0789,26.93,CP Optimizer


### Đọc bảng trên

**CP-SAT duyệt ít nhánh hơn ở gần như mọi bài** — có bài ít hơn 600 lần. Nguyên
nhân là kiến trúc lai SAT: nó học mệnh đề xung đột (clause learning) nên mỗi lần
thất bại đều cắt được cả một vùng lớn của không gian tìm kiếm. CP Optimizer duyệt
ồ ạt với chi phí mỗi nhánh rất rẻ.

**Nhưng ít nhánh không đồng nghĩa nhanh hơn.** Bài 3.2 là phản ví dụ: CP-SAT duyệt
ít hơn 28 lần số nhánh mà vẫn **về sau**. Mỗi nhánh của CP-SAT đắt hơn nhiều.

Cần đọc bảng này **kèm điều kiện về quy mô**: năm bài mà CP-SAT thắng đều giải
xong dưới 1 giây — ở cỡ đó chênh lệch vài chục mili giây không nên ngoại suy. Bài
duy nhất đủ lớn để thời gian có ý nghĩa (3.2, 106 khoá học, 449 biến) thì **CP
Optimizer thắng**.

> **Cảnh báo đơn vị:** `fails` của CP Optimizer và `conflicts` của CP-SAT đếm hai
> thứ khác nhau — nhánh chết so với mệnh đề học được. Chỉ so được trong cùng một
> engine, không so chéo.

## 4. Đối chiếu năng lực — cái gì có sẵn ở đâu

Bảng này tổng hợp từ những gì **thực sự gặp phải** khi cài đặt 6 bài, không phải
từ tài liệu quảng cáo. Mỗi dòng đều có một bài làm chứng.

| Khả năng | CP Optimizer | CP-SAT | Chứng cứ |
|---|---|---|---|
| `allDifferent` | ✅ | ✅ `add_all_different` | 1.1, 1.2, 3.1 |
| `allowedAssignments` (quan hệ bảng) | ✅ | ✅ `add_allowed_assignments` | 3.1 |
| `inverse` | ✅ | ✅ `add_inverse` — buộc miền `0..k-1` | 3.1 |
| `count` | ✅ | ❌ phải 1-hot, **+180 biến bool** ở $n=6$ | 3.1 |
| biến `interval`, `noOverlap` | ✅ | ✅ `new_interval_var`, `add_no_overlap` | 2.1, 3.2 |
| `alternative` — chọn tài nguyên, tự đồng bộ | ✅ | ❌ dựng tay literal hiện diện từng cặp | 3.2 |
| `forbidExtent` + hàm bậc thang — lịch bận | ✅ | ❌ bung thành phép tuyển, **1 bool mỗi (việc, thời điểm bận)** | 3.2 |
| dùng ràng buộc **như biểu thức** (reify tại chỗ) | ✅ | ❌ phải vật chất hoá qua biến trung gian | 3.1 |
| `exactly_one` / `at_most_one` ở tầng boolean | ❌ phải viết tổng số học | ✅ nguyên hàm, nạp thẳng thành CNF | 2.2 |
| `AddAutomaton`, `AddCircuit`, module routing | ❌ | ✅ | (ngoài phạm vi 6 bài) |

**Hai chiều thiếu hụt đối xứng nhau, và đó là lời giải thích cho toàn bộ trục engine:**

- Ở bài **3.2**, CP Optimizer có `alternative` và `forbidExtent` nên ràng buộc khả
  dụng **không sinh thêm một biến quyết định nào**; CP-SAT phải bù bằng biến bool.
  CP Optimizer thắng.
- Ở bài **2.2**, CP-SAT có `add_exactly_one` nạp thẳng thành mệnh đề CNF cho CDCL
  nên chứng minh tối ưu với **0 conflict**; `docplex.cp` không có tương đương ở
  tầng boolean, phải viết `sum(...) == 1` rồi lan truyền bằng suy luận khoảng, tốn
  418 fails. CP-SAT thắng.

⇒ Kết luận đúng **không phải** "engine nào mạnh hơn", mà là: **mỗi engine thắng ở
lớp bài mà cấu trúc dữ liệu lõi của nó phục vụ.** Và chỗ trống trong kho ví dụ
chính thức của mỗi bên chỉ về đúng cùng hướng đó.

## 5. Bài 3.2 — vì sao báo cáo cần đủ ba chiều

Đây là bài duy nhất có benchmark định lượng, và là bằng chứng rõ nhất cho luận
điểm phương pháp ở notebook mở đầu.

Nếu chỉ so hai chiều OPL và OR-Tools, số liệu nói **CP-SAT nhanh hơn 3.8×**:

| Chiều | Engine | Mã hoá | Thời gian |
|---|---|---|---|
| OPL | CP Optimizer | đếm có điều kiện | 7.74 s |
| OR-Tools | CP-SAT | interval | 2.04 s |

Kết luận đó **sai**. Thêm điểm đo thứ ba — cùng engine CP Optimizer nhưng dùng mã
hoá interval — thì bức tranh đảo ngược:

In [5]:
import pandas as pd
b = pd.read_csv("../results/bench.csv")
(b[b.ok].groupby(["config", "dimension", "engine", "encoding"], sort=False)
   [["objective", "solve_time_s", "branches", "fails"]].median().reset_index())

,config,dimension,engine,encoding,objective,solve_time_s,branches,fails
0,small,opl,CP Optimizer,đếm có điều kiện,20.0,0.3240,30412.0,12917.0
1,small,docplexcp,CP Optimizer,interval,20.0,0.4470,75025.0,33825.0
2,small,ortools,CP-SAT,interval,20.0,0.0704,575.0,6.0
3,large,opl,CP Optimizer,đếm có điều kiện,44.0,17.1720,216066.0,96460.0
4,large,docplexcp,CP Optimizer,interval,44.0,0.5840,34194.0,7037.0
5,large,ortools,CP-SAT,interval,44.0,2.4620,7095.0,185.0
6,large+avail,opl,CP Optimizer,đếm có điều kiện,47.0,7.7360,125151.0,55086.0
7,large+avail,docplexcp,CP Optimizer,interval,47.0,1.4130,154818.0,59201.0
8,large+avail,ortools,CP-SAT,interval,47.0,2.0389,5399.0,141.0


Tách ra thành hai phép so, mỗi phép đổi **đúng một** biến số:

| Phép so | Đổi cái gì | Kết quả (bộ `large+avail`) |
|---|---|---|
| OPL → DOcplex.cp | **mã hoá** (đếm → interval), engine giữ nguyên | 7.74 s → 1.41 s, **nhanh 5.5×** |
| DOcplex.cp → OR-Tools | **engine** (CP Optimizer → CP-SAT), mã hoá giữ nguyên | 1.41 s → 2.04 s, **chậm 1.4×** |

Phần lớn khoảng cách 3.8× ban đầu đến từ **cách mã hoá**, không phải từ engine. Và
khi so công bằng trên cùng cách mã hoá thì **CP Optimizer mới là bên nhanh hơn** —
ngược hẳn kết luận rút ra từ hai chiều.

Đây chính là điều mà so sánh trực tiếp "OPL vs OR-Tools" không làm được, và là lý
do chiều DOcplex.cp tồn tại trong báo cáo này.

## 6. Trần Community Edition — đã đo được ở đâu

CP Optimizer bản Community chỉ nhận bài có không gian tìm kiếm tới $2^{1000}$.
Đây là giới hạn **giấy phép**, không phải giới hạn năng lực engine.

| Bài | log₂ không gian tìm kiếm | CP Optimizer | CP-SAT |
|---|---|---|---|
| 1.2 N-Queens, $N=8$ | 24.0 | ✅ | ✅ |
| 1.2 N-Queens, $N=200$ | ≈1529 | ❌ `FATAL[ENGINE_001]` | ✅ |
| 3.1 Sports, $n=6$ | 510.6 | ✅ | ✅ |
| 3.1 Sports, $n=8$ và $n=10$ | >1000 | ❌ `FATAL[ENGINE_001]` | ✅ giải tối ưu cả hai |
| 3.2 Timetable, bộ `large` | 786.4 | ✅ | ✅ |
| 3.2 Timetable, `large` + khả dụng | **780.3** | ✅ | ✅ |

Ba điều rút ra:

1. **$n=10$ của bài 3.1 chính là cỡ dữ liệu trong bản gốc IBM** — bản Community
   không chạy nổi ví dụ chính thức của chính IBM. Muốn dùng đúng cỡ gốc thì cần
   license academic (miễn phí qua IBM Academic Initiative).
2. **Thêm ràng buộc làm không gian tìm kiếm GIẢM** (786.4 → 780.3 ở bài 3.2), dù
   số ràng buộc gần gấp đôi. Trần tính theo *không gian tìm kiếm*, không theo *số
   ràng buộc* — nên ràng buộc thực tế hơn không làm bài toán chạm trần sớm hơn.
3. **Cách mã hoá quyết định có lọt trần hay không.** Với mã hoá nhị phân,
   $\log_2$ không gian bằng đúng số biến bool; bài 3.2 mã hoá kiểu đó sẽ cần
   ~2800 và vượt trần, trong khi mã hoá bằng biến nguyên chỉ tốn 780.

## 7. Kết luận

**Về phương pháp.** Ba chiều là số tối thiểu để tách được nguyên nhân. Bài 3.2 cho
thấy so hai chiều dẫn tới kết luận ngược hẳn sự thật — và không có cách nào phát
hiện điều đó nếu chỉ nhìn hai cột số liệu.

**Về ngôn ngữ mô hình hoá.** Ngôn ngữ chỉ ảnh hưởng tới hiệu năng khi nó **ép viết
một mô hình toán khác đi**. Khác biệt thuần cú pháp dừng ở mức dễ đọc. OPL gọn hơn
ở ràng buộc quan hệ và tupleset; `docplex.cp` gọn hơn ở chỗ nhận thẳng biểu thức
Python vào ràng buộc toàn cục.

**Về engine.** Không có bên nào thắng toàn diện. CP-SAT thắng ở bài tổ hợp/boolean
nhờ học mệnh đề và các nguyên hàm boolean; CP Optimizer thắng ở bài lập lịch quy
mô lớn nhờ `alternative`, `forbidExtent` và bộ lan truyền chuyên cho biến interval.
Sáu bài của báo cáo cho 5–1 nghiêng về CP-SAT, nhưng năm bài đó đều giải xong dưới
1 giây; bài duy nhất đủ lớn để thời gian có ý nghĩa thì CP Optimizer thắng.

**Về cách chọn công cụ.** Câu hỏi đúng không phải "engine nào tốt hơn" mà là *"bài
của tôi có cấu trúc gì"*:

| Nếu bài toán của bạn… | Nghiêng về |
|---|---|
| nhiều ràng buộc boolean, đếm, phủ, tổ hợp | **CP-SAT** |
| lập lịch có tài nguyên, lịch bận, chọn máy/người | **CP Optimizer** |
| cần routing, automaton, chuỗi trạng thái | **CP-SAT** (có sẵn `AddCircuit`, `AddAutomaton`) |
| cần chạy quy mô công nghiệp trên bản miễn phí | **CP-SAT** (không giới hạn giấy phép) |

**Một cảnh báo khi đọc lại báo cáo này.** Mọi số liệu đo trên **một máy, một bộ dữ
liệu, bản Community**. Chúng đủ để chỉ ra *cơ chế* — vì sao mã hoá này tốn nhiều
nhánh hơn, vì sao engine kia cần thêm biến phụ — nhưng không đủ để xếp hạng hiệu
năng tổng quát. Phần có giá trị lâu dài là các **cơ chế** đã truy được nguyên nhân
và kiểm chứng bằng thí nghiệm đối chứng, không phải các con số giây.